# Final Six-Owner Model Comparison

This notebook compares the **six student-owned model results** using the ownership mapping currently used by the group.

| Student | IT Number | Model |
|---|---|---|
| Hettiarachchi H.A.A.S | IT25104064 | EfficientNetB0 |
| Thennakoon T.M.A.N | IT25100854 | MLP |
| Senanayake S.M.K.D | IT25100233 | SVM |
| Weerakkody W.W.M.B | IT25101776 | Custom CNN |
| Malintha K.K.I | IT25102346 | EfficientNetB0 |
| Somarathne R.N.N | IT25100295 | DenseNet121 |

Because EfficientNetB0 is assigned to two students, the comparison uses **Owner + Model** as the unique display name.

## Selection Rule

The primary comparison metric is **Validation Macro F1**.

The Official Test set should be used only after variant selection for final reporting.  
Test Accuracy, Precision, Recall and Test Macro F1 are shown for interpretation, not for tuning.

## 1. Upload the Six Final Result CSV Files

In [ ]:
from google.colab import files
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

uploaded = files.upload()

csv_files = {
    name: content
    for name, content in uploaded.items()
    if name.lower().endswith(".csv")
}

print("CSV files uploaded:", len(csv_files))
for name in csv_files:
    print("-", name)

## 2. Normalize Result Columns

In [ ]:
def normalize_result(filename, content):
    df = pd.read_csv(io.BytesIO(content))

    if len(df) == 0:
        raise ValueError(f"{filename} is empty.")

    row = df.iloc[0].to_dict()

    def first_value(names):
        for name in names:
            if name in row and pd.notna(row[name]):
                return row[name]
        return None

    return {
        "Source File": filename,
        "Student": first_value([
            "Student",
            "Model Owner",
            "Owner",
        ]),
        "IT Number": first_value([
            "IT Number",
            "Owner IT Number",
        ]),
        "Model": first_value(["Model"]),
        "Selected Variant": first_value(["Selected Variant"]),
        "Selected Variant Name": first_value([
            "Selected Variant Name"
        ]),
        "Validation F1": first_value(["Validation F1"]),
        "Test Accuracy": first_value(["Test Accuracy"]),
        "Test Precision": first_value([
            "Test Precision",
            "Test Precision Macro"
        ]),
        "Test Recall": first_value([
            "Test Recall",
            "Test Recall Macro"
        ]),
        "Test F1": first_value([
            "Test F1",
            "Test F1 Macro"
        ]),
    }


results = pd.DataFrame([
    normalize_result(name, content)
    for name, content in csv_files.items()
])

numeric_columns = [
    "Validation F1",
    "Test Accuracy",
    "Test Precision",
    "Test Recall",
    "Test F1",
]

for column in numeric_columns:
    results[column] = pd.to_numeric(
        results[column],
        errors="coerce"
    )

results["Owner + Model"] = (
    results["Student"].fillna("Unknown Owner")
    + " — "
    + results["Model"].fillna("Unknown Model")
)

display(results)

## 3. Rank by Validation Macro F1

In [ ]:
ranking = (
    results
    .sort_values(
        "Validation F1",
        ascending=False
    )
    .reset_index(drop=True)
)

ranking.insert(
    0,
    "Rank",
    np.arange(1, len(ranking) + 1)
)

display(
    ranking[
        [
            "Rank",
            "Student",
            "IT Number",
            "Model",
            "Selected Variant",
            "Selected Variant Name",
            "Validation F1",
            "Test Accuracy",
            "Test Precision",
            "Test Recall",
            "Test F1",
        ]
    ]
)

## 4. Validation Macro F1 Chart

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(
    ranking["Owner + Model"],
    ranking["Validation F1"]
)
plt.title("Six Student Models — Validation Macro F1")
plt.ylabel("Validation Macro F1")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Official Test Macro F1 Chart

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(
    ranking["Owner + Model"],
    ranking["Test F1"]
)
plt.title("Six Student Models — Official Test Macro F1")
plt.ylabel("Test Macro F1")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Validation-to-Test Gap

In [ ]:
gap = ranking[
    [
        "Owner + Model",
        "Validation F1",
        "Test F1"
    ]
].copy()

gap["Validation-Test F1 Gap"] = (
    gap["Validation F1"]
    - gap["Test F1"]
)

display(gap)

## 7. Save Final Comparison

In [ ]:
from pathlib import Path

OUT = Path("/content/final_comparison")
OUT.mkdir(parents=True, exist_ok=True)

FINAL_CSV = OUT / "FINAL_6_OWNER_MODEL_COMPARISON.csv"

ranking.to_csv(
    FINAL_CSV,
    index=False
)

print("Saved:", FINAL_CSV)

## 8. Download Final Comparison CSV

In [ ]:
from google.colab import files

files.download(
    str(FINAL_CSV)
)